# 02 — Realtime providers

OpenAI Realtime, Gemini Live, Ultravox, ElevenLabs ConvAI.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? (getpatter as any).VERSION ?? 'unknown';
  console.log(`getpatter ${v}`);
});


### Local mode
Construct a Patter instance with a Twilio carrier.


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({
      accountSid: 'ACtest00000000000000000000000000',
      authToken: 'test',
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Patter local mode constructed OK');
});


### Cloud mode (coming soon)
When `apiKey` is supported, Patter cloud handles telephony. For now, the SDK throws — this cell verifies the guard.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  try {
    new Patter({ apiKey: 'pt_test_xxx' } as any);
    throw new Error('expected error — cloud mode guard missing');
  } catch (e: any) {
    if (e.message?.includes('Cloud') || e.message?.includes('cloud') || e.message?.includes('apiKey')) {
      console.log(`cloud mode guard OK: ${e.message.slice(0, 80)}`);
    } else { throw e; }
  }
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  if (rt.provider !== 'openai_realtime') throw new Error(`expected openai_realtime, got ${rt.provider}`);
  if (cv.provider !== 'elevenlabs_convai') throw new Error(`expected elevenlabs_convai, got ${cv.provider}`);
  console.log(`rt=${rt.provider}  cv=${cv.provider}  pl=${pl.provider}`);
});


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


### Local mode
Construct a Patter instance with a Twilio carrier.


### Cloud mode
Same SDK, just an `apiKey` — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises OpenAI Realtime configuration and related SDK primitives.


### OpenAI Realtime agent: full config


In [ ]:
import { Patter, Twilio, OpenAIRealtime } from "getpatter";
await cell('realtime_agent_config', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const agent = p.agent({
    systemPrompt: 'You are a concise assistant.',
    engine: new OpenAIRealtime({ apiKey: 'sk-test', model: 'gpt-4o-realtime-preview' }),
    voice: 'alloy',
    firstMessage: 'Hello! How can I help?',
    bargeInThresholdMs: 500,
  });
  console.log(`provider: ${agent.provider}  voice: ${agent.voice}`);
  console.log(`firstMessage: ${agent.firstMessage}`);
  if (agent.provider !== 'openai_realtime') throw new Error('wrong provider');
});


### SentenceChunker


In [ ]:
import { SentenceChunker } from "getpatter";
await cell('realtime_chunker', { tier: 1, env }, () => {
  const sc = new SentenceChunker();
  const text = 'The weather is sunny. Temperature is 72F. Great day!';
  const chunks: string[] = [];
  for (const char of text) { chunks.push(...sc.push(char)); }
  chunks.push(...sc.flush());
  console.log(`sentences: ${chunks.length}`);
  chunks.forEach((c, i) => console.log(`  [${i}] ${JSON.stringify(c.trim())}`));
});


### Live: OpenAI Realtime models  *(T3 — requires `OPENAI_API_KEY`)*


In [ ]:
await cell('openai_realtime_live', { tier: 3, required: ['openaiKey'], env }, async () => {
  const resp = await fetch('https://api.openai.com/v1/models', {
    headers: { Authorization: `Bearer ${env.openaiKey}` },
  });
  const data = await resp.json() as { data: Array<{ id: string }> };
  const models = data.data.filter(m => m.id.includes('realtime')).map(m => m.id);
  console.log(`OpenAI realtime models: ${models.slice(0, 5)}`);
});


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.
